### Incremental Data Ingestion

In [0]:
dbutils.widgets.text("src", "")

In [0]:
##Check Value + Call a use
src_value = dbutils.widgets.get("src")
src_value

In [0]:
#Read/Update real time (Stream) data

#1. ดึง File จาก Storage
    #Format option = csv
    #Schema location - ที่ๆจะดึงข้อมูล
    #Schema evolution - ถ้าเกิด schema ไม่ตรงกับที่กำหนด จะแก้แบบไหน
        #rescue -
        #AddNewColumn - 

df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", f"/Volumes/dataengineerflightproject/bronze/bronzevolume/{src_value}/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
    .load(f"/Volumes/dataengineerflightproject/raw/rawvolume/rawdata/{src_value}/")

#Execution - load data from location to query the data Stream with option rescue and checkpoint

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("checkpointLocation", f"/Volumes/dataengineerflightproject/bronze/bronzevolume/{src_value}/checkpoint")\
    .option("path", f"/Volumes/dataengineerflightproject/bronze/bronzevolume/{src_value}/data")\
    .start()

#เขียนขึ้น df เป็นแบบ Delta มีทั้ง Checkpoint Location และ Path 
#ให้ Trigger ทีเดียว - ไม่ให้ใช้ตลอดเวลา / AvalableNow - ถ้าข้อมูลเยอะจะแยกย่อยให้ทีละส่วนกันพัง
#ให้เริ่มเลย (start) ใน Bronze volume

In [0]:
%sql
SELECT * FROM delta.`/Volumes/dataengineerflightproject/bronze/bronzevolume/customers/data`